## Detectando casos de COVID a partir de imagens de Tomografia


Esse é um projeto de pós graduação em inteligência artificial cujo o objetivo deste conjunto de dados é incentivar a pesquisa e o desenvolvimento de métodos de
inteligência artificial capazes de identificar se uma pessoa está infectada pelo SARS-CoV-2 por
meio da análise de suas tomografias computadorizadas.

O dataset em questão está disponível no kaggle e foi coletado de pacientes reais em hospitais
no estado de São Paulo.

Nesse projeto, as principais habilidades a serem exercitadas estão relacionadas ao
Processamento de Imagens e a utilização de modelos de Deep Learning para classificação

fonte de dados: https://www.kaggle.com/datasets/plameneduardo/sarscov2-ctscan-dataset

### Carregamento e Pré-processamento básico para normalizar imagens

In [37]:
# importando as blibliotecas

import os #essa lib vai me ajudar a manipular os arquivos, acessar os diretórios, etc.
from PIL import Image #essa lib vai me ajudar a manipular as imagens
#import cv2 
import numpy as np #essa lib vai me ajudar a manipular os arrays
from sklearn.model_selection import train_test_split #essa lib vai me ajudar a dividir os dados em treino, validação e teste
from sklearn.model_selection import train_test_split, KFold
from tensorflow.keras import layers, models

In [38]:
# Definindo o caminho para a pasta de imagens
covid_dir = r'C:\Users\geova\OneDrive\Ambiente de Trabalho\inteligencia artificial\5 machine learning\detectando casos de covid\COVID'
no_covid_dir = r'C:\Users\geova\OneDrive\Ambiente de Trabalho\inteligencia artificial\5 machine learning\detectando casos de covid\non-COVID'

In [39]:

# definindo um tamanho padrão pois quando tentei converter lá embaixo as listas de imagens para arrays, algumas imagens tinham tamanhos diferentes e isso dá erro no codigo porque o array precisa ser retangular, ou seja, todas as imagens precisam ter o mesmo tamanho
img_altura, img_largura = 150, 150

imagens = [] # para armazenar os dados das imagens (como arrays NumPy)
labels = [] # para armazenar as etiquetas correspondentes (1 para COVID, 0 para não COVID)


In [40]:
# Carregar imagens da pasta COVID
for filename in os.listdir(covid_dir):
    if filename.endswith('.png'): # verifica se no final do arquivo tem .png
        img_path = os.path.join(covid_dir, filename) # cria o caminho completo do arquivo
        try:
            img = Image.open(img_path).convert('RGB') # abre a imagem e converte para RGB porque algumas imagens tinham mais 3 de canais provavelmente RGBA e quanto tem o quarto canal que é o alpha, o PIL não consegue converter para array
            img_redimensionada = img.resize((img_largura, img_altura))  # redimensiona a imagem para o tamanho padrão
            img_array = np.array(img_redimensionada)  # converte a imagem para um array numpy
            imagens.append(img_array)  # adiciona a imagem à lista de imagens
            labels.append(1)  # adiciona o label 1 para indicar que é COVID
        except Exception as e:
            print(f'Erro ao processar a imagem {filename}: {e}')

In [41]:
# Carregar imagens da pasta non-COVID
# aqui não comento linha por linha porque é praticamente a mesma coisa que o código acima, só muda o caminho da pasta e o label que é 0
for filename in os.listdir(no_covid_dir):
    if filename.endswith('.png'):  
        img_path = os.path.join(no_covid_dir, filename)  
        try:
            img = Image.open(img_path).convert('RGB')  
            img_redimensionada = img.resize((img_largura, img_altura))  
            img_array = np.array(img_redimensionada)  
            imagens.append(img_array)  
            labels.append(0)  
        except Exception as e:
            print(f"Erro ao processar a imagem {filename}: {e}")

In [42]:
imagens = np.array(imagens)  # converte a lista de imagens para um array numpy
labels = np.array(labels)  # converte a lista de labels para um array numpy

imagens_normalizadas = imagens.astype('float32') / 255.0 # numero float pra que o ajuste sempre seja feito de pouco em pouco e não como INT que é um número inteiro, e dividindo por 255.0 para garantir que o resultado esteja entre 0 e 1, já que os valores dos pixels vão de 0 a 255 sendo 0 preto e 255 branco
# Se um pixel é 0 (preto), 0 / 255.0 resulta em 0.0.
# Se um pixel é 127.5 (um cinza médio), 127.5 / 255.0 resulta em 0.5.
# Se um pixel é 255 (branco), 255 / 255.0 resulta em 1.0.

print(f"Formato do array de imagens normalizadas: {imagens_normalizadas.shape}") # Exibe quantidade de imagens, altura e largura
print(f"Valores dos pixels após a normalização (exemplo - primeiro pixel da primeira imagem): {imagens_normalizadas[0][0][0]}") # Exibe o valor do primeiro pixel da primeira imagem normalizada

Formato do array de imagens normalizadas: (2481, 150, 150, 3)
Valores dos pixels após a normalização (exemplo - primeiro pixel da primeira imagem): [0.7529412 0.7529412 0.7529412]


### Treinamento do modelo

In [43]:
input_shape = (150, 150, 3) # o modelo em si precisa dessa informação explicitamente definida em sua arquitetura mesmo que já passada lá em cima quando convertemos as imagens para arrays, mas é bom deixar explícito aqui também

In [44]:
# Criando o modelo (função para facilitar a criação em cada fold)
def create_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [45]:
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

In [46]:
accuracy_per_fold = []
loss_per_fold = []

In [47]:
fold_no = 1
for train, val in kf.split(imagens_normalizadas, labels):
    print(f"Fold {fold_no}:")

    model = create_model()

    history = model.fit(imagens_normalizadas[train], labels[train],
                        epochs=10,  
                        verbose=1)

    scores = model.evaluate(imagens_normalizadas[val], labels[val], verbose=0)
    print(f'Score para Fold {fold_no}: Perda = {scores[0]:.4f}; Acurácia = {scores[1]:.4f}%')
    accuracy_per_fold.append(scores[1] * 100)
    loss_per_fold.append(scores[0])

    fold_no = fold_no + 1

Fold 1:
Epoch 1/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 13s 192ms/step - accuracy: 0.5658 - loss: 0.8095
Epoch 2/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 12s 201ms/step - accuracy: 0.7987 - loss: 0.4544
Epoch 3/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 232ms/step - accuracy: 0.8220 - loss: 0.4014
Epoch 4/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 13s 216ms/step - accuracy: 0.8470 - loss: 0.3639
Epoch 5/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 17s 273ms/step - accuracy: 0.9053 - loss: 0.2409
Epoch 6/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 231ms/step - accuracy: 0.9255 - loss: 0.1884
Epoch 7/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 12s 191ms/step - accuracy: 0.9563 - loss: 0.1359
Epoch 8/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 11s 179ms/step - accuracy: 0.9476 - loss: 0.1385
Epoch 9/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 11s 179ms/step - accuracy: 0.9657 - loss: 0.1049
Epoch 10/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 11s 180ms/step - accuracy: 0.9716 - loss: 0.0702
Score para Fold 1: Perda = 0.3150; Acurácia = 0.8692%
Fold 2:
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 13s 179ms/step - accu

In [48]:
# Calcular a média das métricas em todos os folds
print("Resultados da Validação Cruzada:")
print(f"Média da Perda: {np.mean(loss_per_fold):.4f}")
print(f"Média da Acurácia: {np.mean(accuracy_per_fold):.2f}% (+/- {np.std(accuracy_per_fold):.2f})")

Resultados da Validação Cruzada:
Média da Perda: 0.2877
Média da Acurácia: 89.16% (+/- 1.75)


In [49]:
model.save('modelo_covid.h5')